## QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU (BẢO TOÀN 100% DỮ LIỆU GỐC)
Quy trình này được thiết kế dựa trên yêu cầu đặc biệt: **KHÔNG XÓA BẤT KỲ DÒNG HAY CỘT NÀO** của tập dữ liệu.
1. **Missing Values:** Điền khuyết 100% bằng Trung vị (Median) và Giá trị phổ biến (Mode).
2. **Outliers:** Cắt xén (Capping) bằng ngưỡng IQR, giữ nguyên số lượng dòng.
3. **Skewness:** Áp dụng biến đổi toán học (Log, Lũy thừa 3) để kéo về phân phối chuẩn.
4. **Zero Variance & Đa cộng tuyến:** Bỏ qua việc xóa cột để giữ nguyên số lượng đặc trưng (Features) ban đầu.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. ĐỌC DỮ LIỆU
df = pd.read_csv('Agri_Data_Cleaned_No_AP_Ratio.csv')
print(f"Kích thước dữ liệu gốc: {df.shape}")

# Phân loại cột
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'Yield' in numeric_cols:
    numeric_predictors = [c for c in numeric_cols if c != 'Yield']
else:
    numeric_predictors = numeric_cols

# 2. XỬ LÝ MISSING VALUES (Điền khuyết 100%)
missing = df.isnull().mean() * 100
for col in missing[missing > 0].index:
    if col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

# 3. XỬ LÝ NGOẠI LAI (Capping bằng IQR, không xóa dòng)
for col in numeric_predictors:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower, upper=upper)

# 4. XỬ LÝ ĐỘ LỆCH (Skewness, bằng toán học)
for col in numeric_predictors:
    sk = df[col].skew()
    if sk > 1:
        min_val = df[col].min()
        df[col] = np.log1p(df[col] - min_val + 1) if min_val <= 0 else np.log1p(df[col])
    elif sk < -1:
        df[col] = np.power(df[col], 3)

# 5. XUẤT FILE KẾT QUẢ
output_file = 'Agri_Data_Preprocessed_No_Deletion.csv'
df.to_csv(output_file, index=False)

print("\n--- XỬ LÝ HOÀN TẤT ---")
print(f"Kích thước dữ liệu sau xử lý: {df.shape} (SO VỚI GỐC: Dữ liệu được bảo toàn 100%)")
print(f"File dữ liệu sạch đã được lưu tại: {output_file}")
